[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/extensor_representations/extensor_representations.ipynb)

# Extensor representations

A multivector acts on a state by multiplication. The states need not fill the whole algebra: a smaller space can carry the multiplication while still distinguishing every multivector and preserving every product. A projector selects such states, and multiplication on their independent coefficients gives each multivector's representation: a unary extensor from states to states. This notebook builds the representations, prints each blade's, recovers multivectors from their representations, and constructs the smallest faithful real representations for every nondegenerate signature through dimension six.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
from numga import Algebra, NumpyContext
from numga.algebras import STA
from examples.math.extensor_representations import core

In [ ]:
# Two directions squaring to plus one. Every coefficient and operation is exact.
ga = Algebra("x+y+")
mv = ga.exact.multivector
Full = ga.gatype.full()
State = ga.gatype(ga.subspace("1 y"))
state_dimension = len(State.output_subspace)

## 1. Keep a smaller space of states

Multiplying on the right by \((1+x)/2\) selects states that are unchanged by another right multiplication by \(x\). Left multiplication by any multivector preserves that property. These states form a left ideal: they are closed under every left multiplication. The coefficients of \(1\) and \(y\) suffice to describe them, since multiplication by the projector supplies the linked \(x\) and \(xy\) coefficients. Reading a state back out keeps those two coefficients and doubles them, since the projector halved them. Reading out after embedding returns every state, and embedding after reading out returns every element of the ideal. A multivector's representation embeds a state, multiplies it on the left and reads it back out.

In [ ]:
multivector = mv.x + 2 * mv.y + mv.xy

projector = (1 + mv.x) / 2
# Into the ideal: the projector fills in the coefficients of x and xy.
state_embedding = State * projector                                      # [] Full <- State
# Out of the ideal: the coefficients of 1 and y, doubled since the projector halved them.
state_readout = (2 * Full).cast(State)                                   # [] State <- Full
# A multivector's representation: embed the state, multiply on the left, read the state back out.
representations = state_readout(Full * state_embedding)                  # [] State <- (Full, State)

# The readout undoes the embedding, and the embedding undoes the readout on the ideal.
state_round_trip_residual = state_readout(state_embedding) - State       # [] State <- State
ideal = Full * projector                                                 # [] Full <- Full
ideal_round_trip_residual = state_embedding(state_readout(ideal)) - ideal   # [] Full <- Full

# Filling the multivector's slot gives its representation; the State slot stays open.
representation = representations(multivector)                            # [] State <- State

# Read the multivector back out of its representation.
recovered = core.readout(representations, representation, Full)          # [] Full

print("multivector: Full")
print(multivector.coefficient_table())
print("extensor representation: State <- State")
print(representation.coefficient_table())
print("recovered multivector: Full")
print(recovered.coefficient_table())

## 2. Every blade has a representation

The representations leave the multiplying multivector open as well as the state. They have two slots: one for an element of the whole algebra and one for the state it acts on. Filling the first slot gives that multivector's representation; its state slot remains open. Filling both slots applies the representation to a state. The four basis blades give four independent representations, and every multivector gives their weighted sum.

In matrix notation these representations are real \(2\times2\) matrices, and they satisfy \(\rho(ab)=\rho(a)\rho(b)\).

In [ ]:
# Each blade acts by multiplication inside the ideal, followed by the same readout.
# The unary maps have output blades on rows and input blades on columns.
for blade in Full.output_subspace.masks:
    print(f"{ga.blade_name(blade)}:\n{representations(mv.blade(blade)).coefficient_table()}\n")

In [ ]:
state = 1 + 2 * mv.y
other_multivector = 3 + mv.x - mv.y

representation = representations(multivector)                                   # [] State <- State
other_representation = representations(other_multivector)                       # [] State <- State
product_representation = representations(multivector * other_multivector)       # [] State <- State

# Composing the two representations is multiplying in the algebra.
composition_residual = representation(other_representation) - product_representation

print("multivector:")
print(multivector.coefficient_table())
print("extensor representation:")
print(representation.coefficient_table())
print("representation applied to the state:")
print(representation(state).coefficient_table())
print("product:")
print((multivector * other_multivector).coefficient_table())
print("the two representations applied in turn:")
print(representation(other_representation(state)).coefficient_table())
print("product acting once:")
print(product_representation(state).coefficient_table())

## 3. Recover the multivector

A representation retains all the multivector's coefficients, and one contraction reads them out. The coefficient of a blade is how much of the representation is that blade's representation: compose it with the representation of the blade's inverse and trace over the state, divided by the number of state components. The code composes the representation with that of an open multivector and traces; solving against the scalar product then replaces each blade by its inverse, which is the blade up to sign. Leaving both algebra inputs open gives the round trip: the readout after the representations is the identity on the algebra.

This readout is the adjoint of the representations, with maps on states paired by their normalized trace and multivectors by their scalar product. The representations preserve those pairings, so their adjoint is also their left inverse.

In matrix notation, with $\gamma_A$ the matrix of blade $A$ on $d$ state components, the readout of a matrix $M$ is the expansion $a_A = \operatorname{tr}(\gamma_A^{-1} M)/d$, and the round trip reads $\operatorname{tr}(\gamma_A^{-1}\gamma_B)/d = \delta_{AB}$.

In [ ]:
# The representation composed with the representation of each blade's inverse, traced over the state.
recovered = core.readout(representations, representation, Full)          # [] Full
coefficient_residual = recovered - multivector

pairing = Full.scalar_product(Full)                                      # [] Scalar <- (Full, Full)

# Compose two open representations and trace their State slot.
# The normalized result is exactly the scalar product of their algebra inputs.
trace_pairing = representations(Full, representations).trace(0, 3)
round_trip = pairing.solve(trace_pairing / state_dimension)              # [] Full <- Full
round_trip_residual = round_trip - Full

print("multivector:")
print(multivector.coefficient_table())
print("extensor representation:")
print(representation.coefficient_table())
print("recovered multivector:")
print(recovered.coefficient_table())

## 4. Project an arbitrary map

A faithful representation need not fill the space of all maps on its states. In three Euclidean dimensions, eight multivector coefficients act on four real state components; a general linear map there has sixteen coefficients. The same trace readout projects any such map onto the representations: the representations after their adjoint are the orthogonal projection onto them under the trace pairing. A map that is already a representation returns unchanged. For any other map, the remainder pairs to zero with every representation.

In [ ]:
ga = Algebra("x+y+z+")
mv = ga.exact.multivector
Full = ga.gatype.full()
State = ga.gatype.even()

projector = (1 + mv.z) / 2
state_embedding = State * projector                                      # [] Full <- State
state_readout = (2 * Full).cast(State)                                   # [] State <- Full
representations = state_readout(Full * state_embedding)                  # [] State <- (Full, State)

# Keep only the scalar component of a state. This is a valid state map, but it is no
# multivector's representation: every nonzero representation has even rank.
arbitrary_map = State.select[0].cast(State)
projected = core.readout(representations, arbitrary_map, Full)        # [] Full
projected_representation = representations(projected)                           # [] State <- State
remainder = arbitrary_map - projected_representation
projection_residual = representations(Full, remainder).trace(0, 2)

print("State:", State)
print("supplied map:", arbitrary_map.gatype)
print(arbitrary_map.coefficient_table())
print("projected multivector:", projected.gatype)
print(projected.coefficient_table())
print("projected representation:", projected_representation.gatype)
print(projected_representation.coefficient_table())
print("remainder:", remainder.gatype)
print(remainder.coefficient_table())

## 5. Construct the state space for each signature

Several commuting blades that square to plus one give several commuting projectors; each halves the state space. Products of those blades divide the full basis into disjoint groups. Keeping one representative per group gives independent state coordinates, and casting back multiplies by the number of projectors' combined copies. The layout helper selects these blades and coordinates; the extensor construction below is the same for every signature. Where the algebra has two independent blocks, the selection retains both, so neither can disappear from the representations.

In matrix notation these are the smallest faithful real matrix representations. Complex and quaternionic entries are counted by their real state components. The [Clifford Algebra Explorer](https://coreality.cc/clifford/) lists the corresponding algebra classifications and projector constructions.

In [ ]:
# The upper dimension of the signature table.
dimensions = 6
signatures = tuple((positive, dimension - positive)
                   for dimension in range(dimensions + 1)
                   for positive in range(dimension, -1, -1))

print(f"{'signature':12s} {'algebra components':>18s} {'real state components':>23s}")
for positive, negative in signatures:
    ga = Algebra((positive, negative, 0))
    mv = ga.exact.multivector
    Full = ga.gatype.full()
    State, involutions = core.spinor_layout(ga)
    multiplicity = 2 ** len(involutions)

    # The choices are discrete blade data. The projector and all subsequent
    # operations are exact extensors, with no coefficient conversion.
    projector = mv.scalar()
    for blade in involutions:
        projector = projector * (1 + mv.blade(blade)) / 2

    state_embedding = State * projector
    state_readout = (multiplicity * Full).cast(State)
    representations = state_readout(Full * state_embedding)

    print(f"{str((positive, negative)):12s} {len(Full.output_subspace):18d}"
          f" {len(State.output_subspace):23d}")

Both the three-dimensional Euclidean algebra and the spacetime algebra of signature $(3,1)$ act faithfully on four real state components. The blade labels of their generators' representations below name the chosen state coordinates: rows are output components and columns are input components.

The spacetime algebra's four tables are its basis vectors acting on spinor states. Composing two of them is multiplying the two vectors, so each squares to its vector's square and any two distinct ones anticommute. The table above shows the price of the opposite signature, $(1,3)$: its smallest faithful real states have eight components.

In the matrix notation of Dirac's equation these four tables read as gamma matrices in a real, Majorana representation, $\gamma^\mu\gamma^\nu + \gamma^\nu\gamma^\mu = 2\eta^{\mu\nu}$; the Dirac representation, in signature $(1,3)$, uses complex $4\times4$ matrices, eight real components.

In [ ]:
representative_signatures = ("x+y+z+", "x+y+z+t-")

for signature in representative_signatures:
    ga = Algebra(signature)
    mv = ga.exact.multivector
    Full = ga.gatype.full()
    State, involutions = core.spinor_layout(ga)
    multiplicity = 2 ** len(involutions)

    projector = mv.scalar()
    for blade in involutions:
        projector = projector * (1 + mv.blade(blade)) / 2
    state_readout = (multiplicity * Full).cast(State)
    representations = state_readout(Full * (State * projector))

    print(f"Cl{ga.pqr[:2]}")
    for blade in ga.basis_vector_masks:
        print(f"\n{ga.blade_name(blade)}:\n{representations(mv.blade(blade)).coefficient_table()}")
    print()

## 6. Spinors of the spacetime algebra

In the spacetime algebra of signature $(1,3)$, the one the [Dirac example](../../relativity/dirac/dirac.ipynb) uses, a spinor is an even multivector: eight real components. A vector acts on it from the left, with the time axis on the right to keep it even. The spin plane `yx`, acting from the right, squares to minus one and commutes with every such action, so the eight coordinates group into four pairs, each coordinate followed by its image under the spin plane. In that layout the four generators act as follows.

In the matrix notation of Dirac's equation these tables read as the Dirac representation of the gamma matrices with each complex entry $a + ib$ written as the block $\begin{pmatrix} a & -b \\ b & a \end{pmatrix}$: the time table as $\gamma^0 = \operatorname{diag}(1, 1, -1, -1)$ and each spatial table as $\gamma^k = \begin{pmatrix} 0 & \sigma_k \\ -\sigma_k & 0 \end{pmatrix}$, with the Pauli matrices $\sigma_k$; the spin plane reads as the imaginary unit.

In [ ]:
ga = STA
mv = ga.exact.multivector
spin_plane = mv.yx
# Even multivectors, each coordinate followed by its image under the spin plane.
Spinor = ga.gatype(ga.subspace("1 yx zx zy tz tzyx tx ty"))
complex_unit = (Spinor * spin_plane).cast(Spinor)                         # [] Spinor <- Spinor

print("spin plane, from the right:")
print(complex_unit.coefficient_table())
for blade in ga.basis_vector_masks:
    # The generator from the left and the time axis from the right.
    action = (mv.blade(blade) * Spinor * mv.t).cast(Spinor)               # [] Spinor <- Spinor
    print(f"\n{ga.blade_name(blade)}:\n{action.coefficient_table()}")

With complex coefficients the pairing is built into the numbers. Right multiplication by the spin plane squares to minus one, so `(1 - 1j * yx) / 2` is a projector: it keeps the spinors on which the spin plane acts as multiplication by `1j`. As in section 1, the coefficients of `1 zx tz tx` describe them, reading out doubles those coefficients, and each generator becomes a map on four complex components.

In the matrix notation of Dirac's equation these four tables read as the gamma matrices of the Dirac representation, $\gamma^0 = \operatorname{diag}(1, 1, -1, -1)$ and $\gamma^k = \begin{pmatrix} 0 & \sigma_k \\ -\sigma_k & 0 \end{pmatrix}$, entry for entry.

In [ ]:
complex_mv = NumpyContext(ga, dtype=complex).multivector
Even = ga.gatype.even()
DiracSpinor = ga.gatype(ga.subspace("1 zx tz tx"))
# The spinors on which the spin plane acts as multiplication by 1j.
projector = (1 - 1j * complex_mv.yx) / 2
spinor_embedding = DiracSpinor * projector                                # [] Even <- DiracSpinor
spinor_readout = (2 * Even).cast(DiracSpinor)                             # [] DiracSpinor <- Even

print("spin plane, from the right:")
print(spinor_readout(spinor_embedding * complex_mv.yx).coefficient_table())
for blade in ga.basis_vector_masks:
    # The generator from the left and the time axis from the right, as before.
    action = spinor_readout(complex_mv.blade(blade) * spinor_embedding * complex_mv.t)   # [] DiracSpinor <- DiracSpinor
    print(f"\n{ga.blade_name(blade)}:\n{action.coefficient_table()}")

In [ ]:
# checks
print("State readout after embedding less the identity:\n" + state_round_trip_residual.coefficient_table())
print("Embedding after state readout, on the ideal, less the identity:\n" + ideal_round_trip_residual.coefficient_table())
print("Composition less multiplication:\n" + composition_residual.coefficient_table())
print("Recovered multivector less its input:\n" + coefficient_residual.coefficient_table())
print("Recovery less the identity:\n" + round_trip_residual.coefficient_table())
print("Trace pairing of the projection remainder:\n" + projection_residual.coefficient_table())
# The spin plane squares to minus one and commutes with every generator's action on spinors.
print("Complex unit squared, plus the identity:\n" + (complex_unit(complex_unit) + Spinor).coefficient_table())
for blade in ga.basis_vector_masks:
    action = (mv.blade(blade) * Spinor * mv.t).cast(Spinor)
    print(f"{ga.blade_name(blade)} action and complex unit, less the reverse order:\n"
          + (action(complex_unit) - complex_unit(action)).coefficient_table())
# Reading out undoes embedding, and on the projected spinors the spin plane is multiplication by 1j.
print("Spinor readout after embedding less the identity:\n"
      + (spinor_readout(spinor_embedding) - DiracSpinor).coefficient_table())
print("Spin plane on projected spinors, less 1j:\n"
      + (spinor_readout(spinor_embedding * complex_mv.yx) - 1j * spinor_readout(spinor_embedding)).coefficient_table())